# 07. ANFIS Training (MLP Surrogate)

### Objective: Train MLP surrogate model on 6-feature ANFIS inputs.

### Output Artifacts: Trained model parameters (saved to `../../data/models/anfis_params.json`)

In [1]:
import pandas as pd
import numpy as np
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
import json
import os

INPUT_PATH = '../../data/processed/6_anfis_dataset.csv'
MODEL_DIR = '../../data/models'
os.makedirs(MODEL_DIR, exist_ok=True)

print("Libraries imported")

Libraries imported


In [2]:
# Load data
df = pd.read_csv(INPUT_PATH)
print(f"Loaded {len(df)} rows")

Loaded 3240 rows


In [3]:
# Train/Test Split
feature_cols = [
    'soft_combat', 'soft_collect', 'soft_explore',
    'delta_combat', 'delta_collect', 'delta_explore'
]

X = df[feature_cols].values
y = df['target_multiplier'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training with {len(feature_cols)} features")

Training with 6 features


In [4]:
# Train MLP
print("Training MLP...")
model = MLPRegressor(hidden_layer_sizes=(16, 8), activation='relu', max_iter=500, random_state=42)
model.fit(X_train, y_train)
print("Training complete")

Training MLP...
Training complete


In [5]:
# Save params
train_mae = mean_absolute_error(y_train, model.predict(X_train))
test_mae = mean_absolute_error(y_test, model.predict(X_test))

params = {
    'architecture': '6-16-8-1',
    'metrics': {'train_mae': train_mae, 'test_mae': test_mae},
    'features': feature_cols
}

with open(os.path.join(MODEL_DIR, 'anfis_params.json'), 'w') as f:
    json.dump(params, f, indent=2)
    
print(f"Model saved. Test MAE: {test_mae:.4f}")

Model saved. Test MAE: 0.0102


In [6]:
# FINAL EXPORT — CANONICAL (v2.2)
import json
from sklearn.metrics import r2_score, mean_squared_error

# Compute R² (missing from earlier cells)
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)
train_mse = mean_squared_error(y_train, y_train_pred)
test_mse = mean_squared_error(y_test, y_test_pred)

print('='*60)
print('TRAINING COMPLETE - OPTION B CANONICAL')
print('='*60)
print(f'Train R²:  {train_r2:.4f}')
print(f'Test R²:   {test_r2:.4f}')
print(f'Train MAE: {train_mae:.6f}')
print(f'Test MAE:  {test_mae:.6f}')
print('='*60)

# Export full MLP weights
export_data = {
    'architecture': {
        'input_size': 6,
        'hidden_layers': [16, 8],
        'output_size': 1,
        'activation': 'relu',
        'output_activation': 'linear'
    },
    'feature_order': feature_cols,
    'weights': [w.tolist() for w in model.coefs_],
    'biases': [b.tolist() for b in model.intercepts_],
    'training_metrics': {
        'train_mae': float(train_mae),
        'test_mae': float(test_mae),
        'train_mse': float(train_mse),
        'test_mse': float(test_mse),
        'train_r2': float(train_r2),
        'test_r2': float(test_r2),
        'num_iterations': model.n_iter_,
        'num_samples': len(X)
    },
    'version': 'v2.2-optionB-canonical',
    'status': 'ACTIVE'
}

save_path = '../../data/models/anfis_mlp_weights.json'
print(f'Saving to: {save_path}')

with open(save_path, 'w') as f:
    json.dump(export_data, f, indent=2)

print('✓ Model exported successfully')
print('='*60)

TRAINING COMPLETE - OPTION B CANONICAL
Train R²:  0.9550
Test R²:   0.9524
Train MAE: 0.010339
Test MAE:  0.010166
Saving to: ../../data/models/anfis_mlp_weights.json
✓ Model exported successfully
